# Trabajo Práctico 2 - Grupo 02

### Modelo Bayes Naive

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo es establecer un baseline con Multinomial Bayes Naive sobre dos representaciones de texto, Bag of Words y TF-IDF, con búsqueda de hiperparámetros.

NB es el modelo canónico para clasificación de texctos, es rápido de entrenar y transparente, ya que los pesos son log-probabilidades por palabra y clase.

Cualquier modelo mas sofisticado debe superar el F1-macro de NB, si no lo hace hay un bug.

## Importación e instalación de dependencias


In [1]:
!pip install -q spacy
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 78.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

In [3]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [4]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [5]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N2: Bayes Naive + TF-IDF con priors uniformes

Los priors son la probabilidad que el modelo le asigna a cada clase antes de ver el texto.

Naive Bayes clasifica calculando, para cada clase, la probabilidad de que un texto pertenezca a ella. Esa probabilidad tiene dos partes que se multiplican:

P(clase | texto) = P(texto | clase) × P(clase)

P(texto | clase) es la probabilidad de ver esas palabras dado que la reseña es, por ejemplo, negativa.

P(clase) es el prior, qué tan probable es esa clase en general, sin importar el texto.

Con priors por defecto (class_prior=None), los aprende del data: negativa 40%, neutra 20%, positiva 40%. Entonces antes de leer una sola palabra, el modelo ya cree que una reseña tiene el doble de chances de ser negativa o positiva que neutra.

In [6]:
# Agrupa vectorizador y modelo en un solo objeto
pipe = Pipeline([
    ("tfidf", make_tfidf()),
    ("nb",    MultinomialNB(alpha=1.0, class_prior=[1/3, 1/3, 1/3]))])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_val)
evaluate("nb_tfidf_v2", y_val, y_pred, hyperparams={"alpha": 1.0, "vectorizer": "TF-IDF", "class_prior": "uniform"})

# Reentrenar en train completo y generar submission
pipe.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
save_model(pipe, "nb_tfidf_v2")
make_submission(pipe, pd.DataFrame({"id": test["id"], "text": X_test}), "submission_nb_tfidf_v2.csv");


=== nb_tfidf_v2 ===
Hiperparámetros: {'alpha': 1.0, 'vectorizer': 'TF-IDF', 'class_prior': 'uniform'}

F1-macro:  0.6474
Precision: 0.6480
Recall:    0.6474
Accuracy:  0.7030

              precision    recall  f1-score   support

    negativa     0.7488    0.7789    0.7636      4080
      neutra     0.3978    0.3691    0.3829      2040
    positiva     0.7974    0.7941    0.7958      4080

    accuracy                         0.7030     10200
   macro avg     0.6480    0.6474    0.6474     10200
weighted avg     0.6981    0.7030    0.7003     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      3178     661       241
neutra         705     753       582
positiva       361     479      3240
Modelo guardado → models/nb_tfidf_v2.joblib
Predicción guardada → submissions/submission_nb_tfidf_v2.csv  (8500 predicciones)
Distribución: clase 0: 41.5%, clase 1: 18.1%, clase 2: 40.4%


Forzar priors uniformes mejora el F1-macro de 0.5753 a 0.6474, una ganancia de casi 7 puntos que viene casi completamente de la clase neutra: su F1 sube de 0.15 a 0.38. La matriz de confusión lo confirma, el modelo ahora clasifica correctamente 753 reseñas neutras en lugar de 180, aunque a costa de introducir más confusión en las otras dos clases (negativa baja de 0.78 a 0.76 y positiva se mantiene en 0.79). La distribución de predicciones sobre el test (41.5% / 18 1% / 40.4%) es también mucho más razonable que la del experimento anterior (50% / 3% / 47%), lo que indica que el modelo ya no está ignorando sistemáticamente la clase minoritaria. El principal error restante es la confusión entre neutra y las otras dos clases: 705 neutras clasificadas como negativas y 582 como positivas, lo cual es esperable dado que las reseñas neutras comparten vocabulario con ambos extremos.